In [9]:
import sys
import os
import numpy as np
import pickle
import time

sys.path.insert(0, './GWAN-main/GWAN_master')
from datasets.RadiusGraph import RadiusGraph

In [10]:
DATA_DIR        = 'v2_1024_FINAL22'
SIGNAL_SIZE     = 1024
NODES_PER_GRAPH = 10
NUM_AUG_COPIES  = 10

PKL_OUTPUT = './GWAN-main/GWAN_master/datasets/HypoidGear_BATCH1_2.pkl'

In [11]:
X_all    = np.load(f'{DATA_DIR}/X_batch1.npy').astype(np.float32)
y_all    = np.load(f'{DATA_DIR}/y_batch1.npy')
rpm_all  = np.load(f'{DATA_DIR}/rpm_batch1.npy')
load_all = np.load(f'{DATA_DIR}/load_batch1.npy')

TARGET_RPM  = 1000
TARGET_LOAD = 50

mask = (rpm_all == TARGET_RPM) & (load_all == TARGET_LOAD)
X_all = X_all[mask]
y_all = y_all[mask]

assert X_all.shape[1] == SIGNAL_SIZE
print(f'Batch 1 filtered to {TARGET_RPM} RPM / {TARGET_LOAD} Nm: {X_all.shape}')
for cls, cnt in zip(*np.unique(y_all, return_counts=True)):
    print(f'  Class {cls}: {cnt}')

Batch 1 filtered to 1000 RPM / 50 Nm: (1750, 1024)
  Class 0: 250
  Class 1: 250
  Class 2: 250
  Class 3: 250
  Class 4: 250
  Class 5: 250
  Class 6: 250


In [12]:
def zscore(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True)
    std[std < 1e-8] = 1e-8
    return ((X - mean) / std).astype(np.float32)

In [16]:
def build_graphs_augmented(X, y, nodes_per_graph, n_aug_copies):
    all_graphs = []

    for cls in sorted(np.unique(y)):
        mask = y == cls
        X_cls = X[mask] 

        t0 = time.time()

        X_norm = zscore(X_cls)
        graphs = RadiusGraph(nodes_per_graph, X_norm, cls, 'Graph')
        all_graphs.extend(graphs)
        n_base = len(graphs)

        for copy_i in range(n_aug_copies):
            stds  = X_cls.std(axis=1, keepdims=True)
            noise = np.random.laplace(loc=0, scale=stds, size=X_cls.shape).astype(np.float32)
            X_aug = zscore(X_cls + noise)
            graphs = RadiusGraph(nodes_per_graph, X_aug, cls, 'Graph')
            all_graphs.extend(graphs)

        elapsed = time.time() - t0
        total_cls = n_base * (1 + n_aug_copies)
        print(f'  Gedimas {cls}: {len(X_cls)} segmentu → {n_base} baziniu grafu '
              f'× {1 + n_aug_copies} kopiju = {total_cls} grafai ({elapsed:.1f}s)')

    perm = np.random.permutation(len(all_graphs))
    return [all_graphs[i] for i in perm]


print('Konstruojami 1 bandymo grafai is segmentu (su augmentuotais segmentais)...')
np.random.seed(42)
train_graphs = build_graphs_augmented(X_all, y_all, NODES_PER_GRAPH, NUM_AUG_COPIES)
print(f'\nIs viso grafu: {len(train_graphs)}')

Konstruojami 1 bandymo grafai is segmentu (su augmentuotais segmentais)...
  Gedimas 0: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 1: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 2: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 3: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 4: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 5: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)
  Gedimas 6: 250 segmentu → 25 baziniu grafu × 11 kopiju = 275 grafai (0.8s)

Is viso grafu: 1925


In [17]:
from sklearn.model_selection import train_test_split

graph_labels = [int(g.y) for g in train_graphs]

train_graphs, test_graphs = train_test_split(
    train_graphs, test_size=0.2, stratify=graph_labels, random_state=42
)

print(f'\nMOkymo aibes grafai: {len(train_graphs)}')
print(f'Testavimo aibes grafai:  {len(test_graphs)}')


MOkymo aibes grafai: 1540
Testavimo aibes grafai:  385


In [18]:
with open(PKL_OUTPUT, 'wb') as f:
    pickle.dump(train_graphs, f)
print(f'Issaugota {len(train_graphs)} mokymo aibes grafu → {PKL_OUTPUT}')

PKL_TEST = PKL_OUTPUT.replace('.pkl', '_test.pkl')
with open(PKL_TEST, 'wb') as f:
    pickle.dump(test_graphs, f)
print(f'Issaugota {len(test_graphs)} testavimo aibes grafu → {PKL_TEST}')

Issaugota 1540 mokymo aibes grafu → ./GWAN-main/GWAN_master/datasets/HypoidGear_BATCH1_2.pkl
Issaugota 385 testavimo aibes grafu → ./GWAN-main/GWAN_master/datasets/HypoidGear_BATCH1_2_test.pkl


In [12]:
def build_graphs_no_aug(X, y, nodes_per_graph):
    all_graphs = []
    for cls in sorted(np.unique(y)):
        mask = y == cls
        X_cls = X[mask]
        t0 = time.time()
        X_norm = zscore(X_cls)
        graphs = RadiusGraph(nodes_per_graph, X_norm, cls, 'Graph')
        print(f'  Gedimas {cls}: {len(X_cls)} segmentai → {len(graphs)} grafai ({time.time()-t0:.1f}s)')
        all_graphs.extend(graphs)
    perm = np.random.permutation(len(all_graphs))
    return [all_graphs[i] for i in perm]


X_test_b2  = np.load(f'{DATA_DIR}/X_test.npy').astype(np.float32)
y_test_b2  = np.load(f'{DATA_DIR}/y_test.npy')
rpm_test   = np.load(f'{DATA_DIR}/rpm_test.npy')
load_test  = np.load(f'{DATA_DIR}/load_test.npy')

mask_b2 = (rpm_test == TARGET_RPM) & (load_test == TARGET_LOAD)
X_test_b2 = X_test_b2[mask_b2]
y_test_b2 = y_test_b2[mask_b2]

print(f'2 bandymas ({TARGET_RPM} sukiai / {TARGET_LOAD} Nm): {X_test_b2.shape}')
for cls, cnt in zip(*np.unique(y_test_b2, return_counts=True)):
    print(f'  Gedimas {cls}: {cnt}')

print('\nKonstruojami 2 bandymo grafai is segmentu (su augmentuotais segmentais)...')
np.random.seed(42)
test_graphs_b2 = build_graphs_no_aug(X_test_b2, y_test_b2, NODES_PER_GRAPH)
print(f'Is viso: {len(test_graphs_b2)}')

PKL_TEST_B2 = PKL_OUTPUT.replace('.pkl', '_test_batch2.pkl')
with open(PKL_TEST_B2, 'wb') as f:
    pickle.dump(test_graphs_b2, f)
print(f'Issaugota {len(test_graphs_b2)} grafu → {PKL_TEST_B2}')

Batch 2 (1000 RPM / 50 Nm): (1729, 1024)
  Class 0: 247
  Class 1: 247
  Class 2: 247
  Class 3: 247
  Class 4: 247
  Class 5: 247
  Class 6: 247

Building Batch 2 test graphs...
  Class 0: 247 segments → 24 graphs (0.1s)
  Class 1: 247 segments → 24 graphs (0.1s)
  Class 2: 247 segments → 24 graphs (0.1s)
  Class 3: 247 segments → 24 graphs (0.1s)
  Class 4: 247 segments → 24 graphs (0.1s)
  Class 5: 247 segments → 24 graphs (0.1s)
  Class 6: 247 segments → 24 graphs (0.1s)
Total: 168
Saved 168 graphs → ./GWAN-main/GWAN_master/datasets/HypoidGear_test_batch2.pkl
